## Deepfake Detection with Transfer Learning

### Project Summary
This project builds an image classification pipeline for deepfake detection using pretrained visual embeddings and a neural network classifier.

### Goals
- Extract image embeddings using a pretrained ResNet18 backbone
- Train a multilayer perceptron (MLP) on top of frozen image features
- Evaluate model performance using validation splits and cross-validation
- Tune learning rate and regularization
- Generate predictions on a held-out test set

### Methods
The workflow uses transfer learning by passing images through a pretrained ResNet18 model and using the resulting embeddings as features for a downstream classifier. A PyTorch MLP is trained on these embeddings, and hyperparameters are selected through stratified cross-validation.

### Focus
This analysis emphasizes practical image classification, transfer learning, model selection, and end-to-end machine learning workflow design.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
import torch.optim as optim
import csv

In [ ]:
print("Loading dataset...")
X_raw, y_raw = torch.load("hw2_data.pt")

print("X shape:", X_raw.shape)
print("y shape:", y_raw.shape)
print("Unique labels:", torch.unique(y_raw))
print("X dtype:", X_raw.dtype)
print("y dtype:", y_raw.dtype)
print("X min/max:", float(X_raw.min()), float(X_raw.max()))

Loading dataset...
X shape: torch.Size([2000, 3, 32, 32])
y shape: torch.Size([2000])
Unique labels: tensor([0, 1])
X dtype: torch.float32
y dtype: torch.int64
X min/max: -1.0 1.0


In [ ]:
print("Loading pretrained ResNet18...")

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = nn.Identity()

for param in resnet.parameters():
    param.requires_grad = False

resnet.eval()

print("ResNet ready.")

Loading pretrained ResNet18...
ResNet ready.


In [ ]:
def extract_features(images, batch_size=100):
    """
    Pass images through ResNet18 to get 512-dimensional embeddings.
    """
    embeddings = []

    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch = images[i:i+batch_size]
            emb = resnet(batch)
            embeddings.append(emb)

    return torch.cat(embeddings)

print("Extracting embeddings...")
X_emb = extract_features(X_raw)

print("Embedding shape:", X_emb.shape)

Extracting embeddings...
Embedding shape: torch.Size([2000, 512])


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=512, hidden_dim=128, num_classes=2):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.output(x)
        return x

print("MLP defined.")

MLP defined.


In [ ]:
model = MLP()

# takeing a small batch from embeddings
sample_batch = X_emb[:5]

outputs = model(sample_batch)

print("Input shape:", sample_batch.shape)
print("Output shape:", outputs.shape)
print("Raw outputs:")
print(outputs)


Input shape: torch.Size([5, 512])
Output shape: torch.Size([5, 2])
Raw outputs:
tensor([[ 0.0629, -0.0708],
        [ 0.1249, -0.0373],
        [ 0.0507, -0.0166],
        [ 0.1047, -0.0495],
        [ 0.1167,  0.0117]], grad_fn=<AddmmBackward0>)


In [ ]:
# converting embeddings + labels into dataset
dataset = TensorDataset(X_emb, y_raw)

# batching size required by baseline
batch_size = 50

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("DataLoader ready.")

DataLoader ready.


In [ ]:
model = MLP()
criterion = nn.CrossEntropyLoss()

# Pick one learning rate for now (we'll grid search later)
lr = 0.01
weight_decay = 0.0

optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

def accuracy(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

epochs = 5  # just a quick test first

for epoch in range(1, epochs+1):
    total_loss = 0.0
    total_acc = 0.0
    batches = 0

    for xb, yb in loader:
        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy(logits, yb)
        batches += 1

    print(f"Epoch {epoch}: loss={total_loss/batches:.4f}, acc={total_acc/batches:.4f}")

Epoch 1: loss=0.6796, acc=0.5530
Epoch 2: loss=0.6459, acc=0.7120
Epoch 3: loss=0.5985, acc=0.7870
Epoch 4: loss=0.5352, acc=0.8100
Epoch 5: loss=0.4674, acc=0.8340


In [ ]:
from sklearn.model_selection import train_test_split

# Make indices for a stratified split (keeps class balance)
idx = np.arange(len(y_raw))
train_idx, val_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=0,
    stratify=y_raw.numpy()
)

X_train, y_train = X_emb[train_idx], y_raw[train_idx]
X_val, y_val     = X_emb[val_idx], y_raw[val_idx]

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=50, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=200, shuffle=False)

print("Train size:", len(train_idx), "Val size:", len(val_idx))

Train size: 1600 Val size: 400


In [ ]:
model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=0.0)

def eval_accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

epochs = 20

for epoch in range(1, epochs+1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

    # report every epoch (you can change to every 5 later)
    train_acc = eval_accuracy(model, train_loader)
    val_acc = eval_accuracy(model, val_loader)
    print(f"Epoch {epoch:02d}: train_acc={train_acc:.4f}, val_acc={val_acc:.4f}")

Epoch 01: train_acc=0.6656, val_acc=0.6625
Epoch 02: train_acc=0.7412, val_acc=0.7350
Epoch 03: train_acc=0.7825, val_acc=0.7900
Epoch 04: train_acc=0.7944, val_acc=0.7875
Epoch 05: train_acc=0.8106, val_acc=0.8125
Epoch 06: train_acc=0.8281, val_acc=0.8125
Epoch 07: train_acc=0.8319, val_acc=0.8250
Epoch 08: train_acc=0.8506, val_acc=0.8325
Epoch 09: train_acc=0.8569, val_acc=0.8550
Epoch 10: train_acc=0.8650, val_acc=0.8600
Epoch 11: train_acc=0.8725, val_acc=0.8650
Epoch 12: train_acc=0.8806, val_acc=0.8725
Epoch 13: train_acc=0.8806, val_acc=0.8825
Epoch 14: train_acc=0.8881, val_acc=0.8825
Epoch 15: train_acc=0.8931, val_acc=0.8875
Epoch 16: train_acc=0.8888, val_acc=0.8950
Epoch 17: train_acc=0.9031, val_acc=0.9000
Epoch 18: train_acc=0.9056, val_acc=0.9100
Epoch 19: train_acc=0.9075, val_acc=0.9150
Epoch 20: train_acc=0.9125, val_acc=0.9100


In [ ]:
from sklearn.model_selection import StratifiedKFold

lrs = [0.001, 0.01, 0.05, 0.1]
wds = [0.0, 1e-4, 1e-3, 1e-2]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

print("Grid size:", len(lrs) * len(wds), "configs")
print("5-fold CV ready.")

Grid size: 16 configs
5-fold CV ready.


In [ ]:
def train_one_fold(X_train, y_train, X_val, y_val, lr, wd, epochs=50, batch_size=50):
    model = MLP()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=200, shuffle=False)

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    # validation accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    return correct / total

In [ ]:
lr_test = 0.01
wd_test = 0.0

fold_accs = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_emb.numpy(), y_raw.numpy()), start=1):
    X_train = X_emb[train_idx].float()
    y_train = y_raw[train_idx].long()
    X_val = X_emb[val_idx].float()
    y_val = y_raw[val_idx].long()

    acc = train_one_fold(X_train, y_train, X_val, y_val, lr=lr_test, wd=wd_test, epochs=10)
    fold_accs.append(acc)
    print(f"Fold {fold}: val_acc={acc:.4f}")

print("Mean val acc:", float(np.mean(fold_accs)))

Fold 1: val_acc=0.8600
Fold 2: val_acc=0.8550
Fold 3: val_acc=0.8625
Fold 4: val_acc=0.8375
Fold 5: val_acc=0.8600
Mean val acc: 0.8549999999999999


In [ ]:
def train_one_fold(X_train, y_train, X_val, y_val, lr, wd, epochs=50, batch_size=50):
    model = MLP()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=200, shuffle=False)

    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model(xb)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return correct / total

In [ ]:
lrs = [0.001, 0.01, 0.05, 0.1]
wds = [0.0, 1e-4, 1e-3, 1e-2]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

best_mean = -1.0
best_params = None
results = []

for lr in lrs:
    for wd in wds:
        fold_accs = []
        for fold, (train_idx, val_idx) in enumerate(skf.split(X_emb.numpy(), y_raw.numpy()), start=1):
            X_train = X_emb[train_idx].float()
            y_train = y_raw[train_idx].long()
            X_val   = X_emb[val_idx].float()
            y_val   = y_raw[val_idx].long()

            acc = train_one_fold(X_train, y_train, X_val, y_val, lr=lr, wd=wd, epochs=50)
            fold_accs.append(acc)

        mean_acc = float(np.mean(fold_accs))
        std_acc  = float(np.std(fold_accs))
        results.append((lr, wd, mean_acc, std_acc))

        print(f"lr={lr:<6} wd={wd:<6} mean_val_acc={mean_acc:.4f} ± {std_acc:.4f}")

        if mean_acc > best_mean:
            best_mean = mean_acc
            best_params = (lr, wd)

print("\nBEST PARAMS:", best_params, "mean_val_acc:", best_mean)

lr=0.001  wd=0.0    mean_val_acc=0.8135 ± 0.0159
lr=0.001  wd=0.0001 mean_val_acc=0.8060 ± 0.0179
lr=0.001  wd=0.001  mean_val_acc=0.8095 ± 0.0120
lr=0.001  wd=0.01   mean_val_acc=0.8175 ± 0.0160
lr=0.01   wd=0.0    mean_val_acc=0.9165 ± 0.0089
lr=0.01   wd=0.0001 mean_val_acc=0.9150 ± 0.0072
lr=0.01   wd=0.001  mean_val_acc=0.9130 ± 0.0113
lr=0.01   wd=0.01   mean_val_acc=0.9100 ± 0.0108
lr=0.05   wd=0.0    mean_val_acc=0.9160 ± 0.0075
lr=0.05   wd=0.0001 mean_val_acc=0.9135 ± 0.0060
lr=0.05   wd=0.001  mean_val_acc=0.9145 ± 0.0144
lr=0.05   wd=0.01   mean_val_acc=0.9110 ± 0.0056
lr=0.1    wd=0.0    mean_val_acc=0.9160 ± 0.0073
lr=0.1    wd=0.0001 mean_val_acc=0.9155 ± 0.0056
lr=0.1    wd=0.001  mean_val_acc=0.9100 ± 0.0120
lr=0.1    wd=0.01   mean_val_acc=0.8360 ± 0.0660

BEST PARAMS: (0.01, 0.0) mean_val_acc: 0.9165000000000001


In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

best_lr, best_wd = 0.05, 1e-4

final_model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(final_model.parameters(), lr=best_lr, weight_decay=best_wd)

full_loader = DataLoader(TensorDataset(X_emb.float(), y_raw.long()),
                         batch_size=50, shuffle=True)

for epoch in range(1, 51):
    final_model.train()
    total_loss = 0.0
    for xb, yb in full_loader:
        optimizer.zero_grad()
        logits = final_model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch in [1, 10, 25, 50]:
        print(f"Epoch {epoch:02d} done, avg_loss={total_loss/len(full_loader):.4f}")

Epoch 01 done, avg_loss=0.6040
Epoch 10 done, avg_loss=0.1843
Epoch 25 done, avg_loss=0.1511
Epoch 50 done, avg_loss=0.0092


In [ ]:
test_obj = torch.load("hw2_test-1.pt")

# checking structure
print("Type:", type(test_obj))

if isinstance(test_obj, (list, tuple)) and len(test_obj) == 2:
    # We need to detect which is images vs ids
    a, b = test_obj

    if a.ndim == 1:
        test_ids = a
        X_test = b
    else:
        X_test = a
        test_ids = b
else:
    raise ValueError("Unexpected test file format")

print("X_test:", X_test.shape)
print("test_ids:", test_ids.shape)
print("X_test min/max:", float(X_test.min()), float(X_test.max()))
print("test_ids min/max:", int(test_ids.min()), int(test_ids.max()))

Type: <class 'list'>
X_test: torch.Size([500, 3, 32, 32])
test_ids: torch.Size([500])
X_test min/max: -1.0 1.0
test_ids min/max: 0 499


In [ ]:
X_test_emb = extract_features(X_test, batch_size=100).float()
print("X_test_emb:", X_test_emb.shape)

final_model.eval()
preds = []
with torch.no_grad():
    test_loader = DataLoader(X_test_emb, batch_size=200, shuffle=False)
    for xb in test_loader:
        logits = final_model(xb)
        preds.append(torch.argmax(logits, dim=1).cpu())

pred_labels = torch.cat(preds).numpy().astype(int)
test_ids_np = test_ids.cpu().numpy().astype(int)

print("Pred unique:", np.unique(pred_labels))
print("Num preds:", len(pred_labels))

X_test_emb: torch.Size([500, 512])
Pred unique: [0 1]
Num preds: 500


In [ ]:
with open("predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["id", "label"])
    for i, lab in zip(test_ids_np, pred_labels):
        w.writerow([int(i), int(lab)])

print("Wrote predictions.csv")

Wrote predictions.csv
